In [1]:
!pip install scikit-learn


In [2]:
import sys
!{sys.executable} -m pip install wfdb scikit-learn

In [3]:
# =====================================================================================
# Project: PTB-XL Metadata Audit  -  v0.5 environment regression check, learning-rate
#          selection (signal_plus_physio only), and the signal_only pipeline sanity gate
# Code version: v0.5
# Date: 2026-10-02
# Purpose:
#   STEP 0  verify pre_registration.json (seal) and dev_protocol.json, load every recipe
#           value FROM dev_protocol.json, verify input CSV hashes, detect existing
#           v0.5_training_details.json / lr_selection.json
#   A  environment regression check (CPU): v0.3's LogisticRegression B1/B3 fold-9
#      macro-AUROC must reproduce after the scipy upgrade
#   -  training/evaluation definition written to PREREG_DIR\v0.5_training_details.json
#      (write-once) BEFORE the first training run
#   B  lr selection: signal_plus_physio, seed 42, every lr in the grid, both
#      architectures, 50 epochs on folds 1-8, ONE evaluation on fold 9 at the end of
#      epoch 50 -> selection rule -> PREREG_DIR\lr_selection.json (write-once)
#   C  pipeline sanity gate: signal_only, resnet1d_wang, seed 42, selected lr; fold-9
#      macro-AUROC >= 0.90
#
# THIS STAGE NEVER TRAINS OR EVALUATES THE signal_plus_all ARM and computes no
# signal_plus_physio vs signal_plus_all comparison (train_run() refuses that arm).
# FOLD 10: no fold-10 record file is opened; signals come only from the v0.4 cache of
# folds 1-9 (sha256-verified). wfdb is not used in v0.5.
#
# REUSE (verbatim from v0.4): helpers, verify_self_sha256, canonical hashing, cache
# loading, lead normalization (folds 1-8), label derivation, metadata matrices, ECGNet
# and all model classes, RandomCropDataset. v0.2.1's fast_auc and LogisticRegression
# setup are reused verbatim for Part A and for every AUROC in this cell.
#
# RE-RUN BEHAVIOUR:
#   - lr_selection.json exists  -> Part B is SKIPPED (no retraining); selected lrs and
#     the table are read from the file.
#   - Part C run files exist     -> the sanity gate is re-evaluated from the SAVED fold-9
#     probabilities; signal_only is NOT retrained (no second draw at the gate).
#
# Outputs: OUT_DIR = D:\ptbxl-metadata-audit\outputs\v0.5
#   v0.5_env_regression.csv
#   per run: v0.5_{arch}_{arm}_seed{seed}_lr{lr}_fold9_probs.csv / _train_loss.csv / _state_dict.pt
#   v0.5_lr_selection_table.csv
#   v0.5_manifest.json (hashes of everything incl. the two new PREREG files)
#   PREREG_DIR: v0.5_training_details.json, lr_selection.json (both write-once)
# =====================================================================================

import os, sys, json, ast, hashlib, platform, time, random, math, re
from datetime import datetime
import numpy as np
import pandas as pd
import scipy
from scipy.stats import rankdata
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

# ------------------------------- configuration ---------------------------------------
DATA_ROOT = r"D:\ptbxl-metadata-audit\data\ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"
OUT_DIR   = r"D:\ptbxl-metadata-audit\outputs\v0.5"
V04_DIR   = r"D:\ptbxl-metadata-audit\outputs\v0.4"
PREREG_DIR = r"D:\ptbxl-metadata-audit\preregistration"
SEAL_PATH = os.path.join(PREREG_DIR, "pre_registration.json")
DEV_PROTOCOL_PATH = os.path.join(PREREG_DIR, "dev_protocol.json")
TRAINING_DETAILS_PATH = os.path.join(PREREG_DIR, "v0.5_training_details.json")
LR_SELECTION_PATH = os.path.join(PREREG_DIR, "lr_selection.json")
CODE_VERSION = "v0.5"
PREFIX = CODE_VERSION + "_"
SUPERCLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]

EXPECTED_SEAL_SHA256 = "5183f54569f3358a5339b16a2c3c3be49f5c7798eabe6b0bb25946119c2ba469"
EXPECTED_DEV_PROTOCOL_SHA256 = "c967ac38c8cc9382bf41fdd3785c8bbd730db196e349846b8052d856bbac5fc6"
EXPECTED_INPUT_HASHES = {
    "ptbxl_database.csv": "7600de9c1b27d181d850b3c6038a35d7c3ddb6bb33b702e3a20252a6859d216b",
    "scp_statements.csv": "ad05b0b1fcae83bb1230755ad9cfc7c96f303feddc08a4a9ad5bdc9ca63bac8f",
}
SIGNALS_NPY = os.path.join(V04_DIR, "v0.4_signals_folds1to9_100hz.npy")
SIGNALS_INDEX = os.path.join(V04_DIR, "v0.4_signals_index.csv")
V04_LEAD_NORM_CSV = os.path.join(V04_DIR, "v0.4_lead_norm_params.csv")
EXPECTED_SIGNALS_NPY_SHA256 = "17558f9d09b39b990d7f3a5ca2ea10e1527980288a49a6828c89b7c3df61745a"
EXPECTED_SIGNALS_INDEX_SHA256 = "06b49ecdafc6a74c9bdbcfa0177bb357eb845ece187b40e3a777496a8cf72203"

EXPECTED_FOLD9_POSITIVES = {"NORM": 955, "MI": 540, "STTC": 528, "CD": 495, "HYP": 268}
EXPECTED_N_RECORDS_FOLDS1TO9 = 19601
EXPECTED_ONEHOT_COUNTS = {"device": 13, "site": 52, "nurse": 14}

# Part A: v0.3 LogisticRegression seed-42 fold-9 macro-AUROC (6 decimals)
V03_LR_MACRO_AUROC = {"B1_physio": 0.688969, "B3_physio_plus_acquisition": 0.697165}
ENV_REGRESSION_TOL = 1e-5

# constants that dev_protocol.json defines (asserted equal to the file; the FILE values are used)
EXPECTED_EPOCHS = 50
EXPECTED_LR_GRID = [1e-3, 3e-3, 1e-2]
EXPECTED_CROP = 250
EXPECTED_EVAL_STRIDE = 125
EXPECTED_WINDOW_STARTS = [0, 125, 250, 375, 500, 625, 750]
EXPECTED_WEIGHT_DECAY = 1e-2
EXPECTED_BATCH_SIZE = {"resnet1d_wang": 128, "inception1d": 128}

SEED = 42
FS = 100
N_SAMPLES = 1000
N_LEADS = 12
NUM_WORKERS = 0                   # Windows + Jupyter
EVAL_BATCH_SIZE = 512             # inference only; does not affect results (model.eval())
SANITY_GATE_THRESHOLD = 0.90
LR_TIE_TOL = 1e-4
SKLEARN_CHECK_TOL = 1e-9

ARCHS = ["resnet1d_wang", "inception1d"]
ARMS = ["signal_only", "signal_plus_physio", "signal_plus_all"]
FORBIDDEN_ARM = "signal_plus_all"
EXPECTED_META_DIM = {"signal_only": 0, "signal_plus_physio": 3, "signal_plus_all": 82}

_root_abs = os.path.abspath(DATA_ROOT)
for _p in (OUT_DIR, PREREG_DIR):
    assert not os.path.abspath(_p).lower().startswith(_root_abs.lower()), \
        "output path is inside the dataset folder - aborting: " + _p

RUN_STARTED = datetime.now().astimezone().isoformat()
OUTPUT_FILES = []        # paths of produced files (for the manifest)
WARNINGS = []


def sha256_of(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def save_csv(obj, name, index=True):
    p = os.path.join(OUT_DIR, name)
    obj.to_csv(p, index=index, encoding="utf-8-sig")
    OUTPUT_FILES.append(p)
    print("[saved] " + p)
    return p


def hdr(t):
    print("\n" + "=" * 100)
    print(t)
    print("=" * 100)


def warn(msg):
    WARNINGS.append(msg)
    print("[WARN] " + msg)


def note(msg):
    print("[NOTE] " + msg)


def _json_default(o):
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return float(o)
    if isinstance(o, np.bool_):
        return bool(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    return str(o)


def to_level_string(series, missing="__MISSING__"):
    """Human-readable, zero-padded categorical level for numeric columns (e.g.
    '007' for site) so sorting and printed feature names stay legible."""
    if pd.api.types.is_numeric_dtype(series):
        nn = series.dropna()
        integral = bool(len(nn) == 0 or np.all(np.isclose(nn.values, np.round(nn.values))))
        if integral:
            width = max(3, len(str(int(nn.max()))) if len(nn) else 3)
            return series.apply(lambda v: missing if pd.isna(v) else str(int(v)).zfill(width))
    return series.apply(lambda v: missing if pd.isna(v) else str(v).strip())


def encode_categorical(level_series, train_mask, col_name):
    """One-hot encode a pre-stringified categorical column. The category list is
    fixed from the training rows only, always includes an explicit '__MISSING__'
    level, and always includes a catch-all '__UNKNOWN__' level for anything seen
    only outside training (e.g. only in fold 9). Both splits therefore produce the
    exact same set of columns."""
    train_vals = level_series[train_mask]
    train_categories = sorted(set(train_vals.unique().tolist()) | {"__MISSING__"})
    known = set(train_categories)
    remapped = level_series.where(level_series.isin(known), other="__UNKNOWN__")
    categories = train_categories + ["__UNKNOWN__"]
    cat = pd.Categorical(remapped, categories=categories)
    dummies = pd.get_dummies(cat)
    dummies.columns = ["{}={}".format(col_name, c) for c in dummies.columns]
    dummies.index = level_series.index
    return dummies, categories


def canonical_json_bytes(d):
    """Canonical serialization used for self_sha256 (see SELF_SHA256_PROCEDURE)."""
    return json.dumps(d, sort_keys=True, ensure_ascii=False, separators=(",", ":"),
                      default=_json_default).encode("utf-8")


SELF_SHA256_PROCEDURE = (
    "Load the JSON file; remove the key 'self_sha256' (the key must be absent, not empty); "
    "serialize the remaining dict with json.dumps(d, sort_keys=True, ensure_ascii=False, "
    "separators=(',', ':')); compute SHA-256 of the UTF-8 bytes; the hex digest must equal "
    "the stored 'self_sha256'. (At write time numpy scalars were converted to plain JSON "
    "types first, so the loaded file hashes identically.)"
)


def verify_self_sha256(path):
    """Return (stored, recomputed, loaded_dict) for a self-hashed JSON document."""
    with open(path, "r", encoding="utf-8") as f:
        d = json.load(f)
    stored = d.pop("self_sha256", None)
    recomputed = hashlib.sha256(canonical_json_bytes(d)).hexdigest()
    d["self_sha256"] = stored
    return stored, recomputed, d


def fast_auc(y, s):
    """Binary AUROC via the Mann-Whitney U statistic (average-rank tie handling).
    Used instead of sklearn.metrics.roc_auc_score for speed inside the bootstrap
    loop; both compute the identical quantity."""
    y = np.asarray(y)
    n1 = int(y.sum())
    n0 = len(y) - n1
    if n1 == 0 or n0 == 0:
        return float("nan")
    ranks = rankdata(np.asarray(s))
    return float((ranks[y == 1].sum() - n1 * (n1 + 1) / 2.0) / (n1 * n0))


def write_once_json(path, doc):
    """Normalize to plain JSON types, add self_sha256, exclusive-create, verify from disk."""
    doc = json.loads(json.dumps(doc, ensure_ascii=False, default=_json_default))
    assert "self_sha256" not in doc
    sha = hashlib.sha256(canonical_json_bytes(doc)).hexdigest()
    doc["self_sha256"] = sha
    with open(path, "x", encoding="utf-8") as f:          # refuses to overwrite
        json.dump(doc, f, ensure_ascii=False, indent=2)
    s_, r_, _ = verify_self_sha256(path)
    assert s_ == r_ == sha, "self_sha256 verification FAILED for " + path
    return sha


def plain(o):
    return json.loads(json.dumps(o, ensure_ascii=False, default=_json_default))


# =====================================================================================
# STEP 0 - INTEGRITY (before anything else; no file is written before this passes)
# =====================================================================================
hdr("STEP 0 - INTEGRITY")

assert os.path.isfile(SEAL_PATH), "sealed pre-registration not found: " + SEAL_PATH
_s, _r, SEAL = verify_self_sha256(SEAL_PATH)
assert _s == _r == EXPECTED_SEAL_SHA256, "pre_registration.json: self_sha256 check FAILED - aborting."
assert SEAL.get("sealed") is True
print("pre_registration.json : self_sha256 {} -> PASS".format(_s))

assert os.path.isfile(DEV_PROTOCOL_PATH), "dev_protocol.json not found: " + DEV_PROTOCOL_PATH
_s, _r, DEV = verify_self_sha256(DEV_PROTOCOL_PATH)
assert _s == _r == EXPECTED_DEV_PROTOCOL_SHA256, "dev_protocol.json: self_sha256 check FAILED - aborting."
assert DEV["parent_seal_self_sha256"] == EXPECTED_SEAL_SHA256
print("dev_protocol.json     : self_sha256 {} -> PASS".format(_s))

# --- recipe values loaded FROM dev_protocol.json, then asserted equal to the constants ---
_rec = DEV["training_recipe"]
EPOCHS = int(_rec["epochs"])
LR_GRID = [float(v) for v in DEV["tuning_grid"]["learning_rate"]]
WEIGHT_DECAY = float(_rec["weight_decay"])
BATCH_SIZE = {k: int(v) for k, v in _rec["batch_size"].items()}
_m = re.search(r"random (\d+)-sample crops", _rec["train_input"])
assert _m, "cannot parse the crop length from dev_protocol train_input"
CROP = int(_m.group(1))
_m = re.search(r"sliding windows of (\d+) samples, stride (\d+) \(window starts \[([0-9, ]+)\]\)", _rec["evaluation"])
assert _m, "cannot parse the evaluation windows from dev_protocol evaluation"
EVAL_WINDOW = int(_m.group(1))
EVAL_STRIDE = int(_m.group(2))
WINDOW_STARTS = [int(v) for v in _m.group(3).split(",")]
AGGREGATION = "mean" if "probability = mean over windows" in _rec["evaluation"] else None
assert "MEAN over sliding windows" in DEV["evaluation_aggregation_note"]
assert _rec["optimizer"] == "AdamW" and _rec["lr_schedule"] == "one-cycle"

assert EPOCHS == EXPECTED_EPOCHS, (EPOCHS, EXPECTED_EPOCHS)
assert LR_GRID == EXPECTED_LR_GRID, (LR_GRID, EXPECTED_LR_GRID)
assert WEIGHT_DECAY == EXPECTED_WEIGHT_DECAY, (WEIGHT_DECAY, EXPECTED_WEIGHT_DECAY)
assert BATCH_SIZE == EXPECTED_BATCH_SIZE, (BATCH_SIZE, EXPECTED_BATCH_SIZE)
assert CROP == EVAL_WINDOW == EXPECTED_CROP, (CROP, EVAL_WINDOW)
assert EVAL_STRIDE == EXPECTED_EVAL_STRIDE and WINDOW_STARTS == EXPECTED_WINDOW_STARTS
assert WINDOW_STARTS == list(range(0, N_SAMPLES - CROP + 1, EVAL_STRIDE))
assert AGGREGATION == "mean"
SELECTION_RULE_TEXT = DEV["selection_rule"]
SANITY_GATE_TEXT = DEV["pipeline_sanity_gate"]
assert "lr is chosen ONLY by the fold-9 macro-AUROC" in SELECTION_RULE_TEXT
assert "macro-AUROC >= 0.90" in SANITY_GATE_TEXT
print("recipe from dev_protocol.json: epochs={} | lr grid={} | weight_decay={} | batch sizes={} | "
      "crop={} | eval windows {} stride {} starts {} | aggregation={} -> all equal the constants: PASS".format(
          EPOCHS, LR_GRID, WEIGHT_DECAY, BATCH_SIZE, CROP, EVAL_WINDOW, EVAL_STRIDE, WINDOW_STARTS, AGGREGATION))

DB_CSV = os.path.join(DATA_ROOT, "ptbxl_database.csv")
SCP_CSV = os.path.join(DATA_ROOT, "scp_statements.csv")
for p in (DB_CSV, SCP_CSV):
    assert os.path.isfile(p), "file not found: " + p
INPUT_HASHES = {os.path.basename(p): sha256_of(p) for p in (DB_CSV, SCP_CSV)}
for k, v in INPUT_HASHES.items():
    assert v == EXPECTED_INPUT_HASHES[k], "input file hash differs from v0.4 for " + k + " - aborting."
print("input CSV hashes      : equal to v0.4 -> PASS")

TRAINING_DETAILS_EXISTS = os.path.exists(TRAINING_DETAILS_PATH)
TRAINING_DETAILS_SHA = None
if TRAINING_DETAILS_EXISTS:
    _s, _r, TRAINING_DETAILS_ON_DISK = verify_self_sha256(TRAINING_DETAILS_PATH)
    assert _s == _r, "v0.5_training_details.json BROKEN (stored != recomputed self_sha256) - aborting."
    TRAINING_DETAILS_SHA = _s
    print("v0.5_training_details.json exists (self_sha256 {}, verified); its contents will be "
          "checked against this code before any use.".format(_s))

LR_SELECTION_EXISTS = os.path.exists(LR_SELECTION_PATH)
LR_SELECTION_SHA = None
SELECTED_LR = None
if LR_SELECTION_EXISTS:
    assert TRAINING_DETAILS_EXISTS, "lr_selection.json exists but v0.5_training_details.json does not - aborting."
    _s, _r, LR_SEL = verify_self_sha256(LR_SELECTION_PATH)
    assert _s == _r, "lr_selection.json BROKEN (stored != recomputed self_sha256) - aborting."
    assert LR_SEL["parent_hashes"]["pre_registration"] == EXPECTED_SEAL_SHA256
    assert LR_SEL["parent_hashes"]["dev_protocol"] == EXPECTED_DEV_PROTOCOL_SHA256
    assert LR_SEL["parent_hashes"]["training_details"] == TRAINING_DETAILS_SHA
    LR_SELECTION_SHA = _s
    SELECTED_LR = {k: float(v) for k, v in LR_SEL["selected_lr"].items()}
    print("lr_selection.json EXISTS (self_sha256 {}, verified) -> selected lr {} -> Part B will be "
          "SKIPPED (no retraining).".format(_s, SELECTED_LR))
else:
    print("lr_selection.json not present -> Part B will run and write it once.")

os.makedirs(OUT_DIR, exist_ok=True)


# =====================================================================================
# DATA (CSV metadata, folds 1-9 only), LABELS, METADATA PREPROCESSING
# =====================================================================================
hdr("DATA, LABELS, METADATA")

df_all = pd.read_csv(DB_CSV, index_col="ecg_id")
df9 = df_all[df_all["strat_fold"] <= 9].copy()
mask_train = df9["strat_fold"] <= 8
mask_eval = df9["strat_fold"] == 9
assert (df9["strat_fold"] == 10).sum() == 0, "fold 10 leaked into the working set - aborting."
n_fold10_in_source = int((df_all["strat_fold"] == 10).sum())
train_patients = set(df9.loc[mask_train, "patient_id"])
eval_patients = set(df9.loc[mask_eval, "patient_id"])
assert train_patients.isdisjoint(eval_patients), \
    "patient-level leakage between train (folds 1-8) and eval (fold 9) - aborting."
current_index = pd.DataFrame({
    "ecg_id": df9.index.to_numpy(),
    "strat_fold": df9["strat_fold"].to_numpy(),
    "patient_id": df9["patient_id"].to_numpy(),
})
N_REC = len(current_index)
assert N_REC == EXPECTED_N_RECORDS_FOLDS1TO9
train_rows = np.where(mask_train.to_numpy())[0]
eval_rows = np.where(mask_eval.to_numpy())[0]
print("records: folds 1-8 = {} | fold 9 = {} | fold-10 records in source CSV, never opened: {}".format(
    len(train_rows), len(eval_rows), n_fold10_in_source))

# --- label derivation (verbatim from v0.2.1) ---
scp = pd.read_csv(SCP_CSV, index_col=0)
diag = scp[scp["diagnostic"].fillna(0) == 1]
code2class = {k: v for k, v in diag["diagnostic_class"].items() if isinstance(v, str)}
parsed = df9["scp_codes"].apply(ast.literal_eval)
rec_classes = parsed.apply(lambda d: {code2class[k] for k in d if k in code2class})
lab = pd.DataFrame({c: rec_classes.apply(lambda s, c=c: int(c in s)) for c in SUPERCLASSES},
                   index=df9.index)
assert (current_index["ecg_id"].to_numpy() == lab.index.to_numpy()).all()
fold9_pos = {c: int(lab.loc[mask_eval, c].sum()) for c in SUPERCLASSES}
assert fold9_pos == EXPECTED_FOLD9_POSITIVES, "fold-9 label counts differ from v0.2.1 - aborting."
print("fold-9 positives:", fold9_pos, "-> PASS")
Y = lab[SUPERCLASSES].to_numpy(dtype=np.float32)
Y_EVAL = {sc: lab.loc[mask_eval, sc].to_numpy() for sc in SUPERCLASSES}

# --- metadata matrices (verbatim from v0.4 Part C) ---
age_raw = df9["age"]
age_sentinel_mask = (age_raw == 300)
n_age_sentinel = int(age_sentinel_mask.sum())
age_clean = age_raw.where(~age_sentinel_mask, np.nan)
age_censored = age_sentinel_mask.astype(float)
age_imputed = age_clean.fillna(90.0)
sex = df9["sex"].astype(float)
print("age: sentinel 300 affects", n_age_sentinel, "records; flagged via age_censored, imputed with 90.")

device_level = to_level_string(df9["device"])
site_level = to_level_string(df9["site"])
nurse_level = to_level_string(df9["nurse"])
device_dummies, device_categories = encode_categorical(device_level, mask_train, "device")
site_dummies, site_categories = encode_categorical(site_level, mask_train, "site")
nurse_dummies, nurse_categories = encode_categorical(nurse_level, mask_train, "nurse")
for name, dmy in [("device", device_dummies), ("site", site_dummies), ("nurse", nurse_dummies)]:
    assert dmy.shape[1] == EXPECTED_ONEHOT_COUNTS[name], \
        "{} one-hot width {} != v0.2.1's {}".format(name, dmy.shape[1], EXPECTED_ONEHOT_COUNTS[name])

age_scaler = StandardScaler().fit(age_imputed[mask_train].to_numpy().reshape(-1, 1))
age_imputed_std = pd.Series(age_scaler.transform(age_imputed.to_numpy().reshape(-1, 1)).ravel(),
                            index=df9.index, name="age_imputed")

B1_cols = ["age_imputed", "age_censored", "sex"]
B2_cols = list(device_dummies.columns) + list(site_dummies.columns) + list(nurse_dummies.columns)
B3_cols = B1_cols + B2_cols

META_FEATURES = pd.concat([age_imputed_std, age_censored.rename("age_censored"), sex.rename("sex"),
                           device_dummies, site_dummies, nurse_dummies], axis=1)
ARM_COLUMNS = {"signal_only": [], "signal_plus_physio": B1_cols, "signal_plus_all": B3_cols}
META = {}
for arm in ARMS:
    cols = ARM_COLUMNS[arm]
    if cols:
        META[arm] = META_FEATURES[cols].to_numpy(dtype=np.float32)
    else:
        META[arm] = np.zeros((N_REC, 0), dtype=np.float32)
    assert META[arm].shape == (N_REC, EXPECTED_META_DIM[arm]), \
        "{}: metadata dim {} != {}".format(arm, META[arm].shape[1], EXPECTED_META_DIM[arm])
    assert np.isfinite(META[arm]).all()
    print("  {:20s} metadata dim = {:3d}{}".format(arm, META[arm].shape[1],
          "" if not cols else "  ({} ... {})".format(", ".join(cols[:3]), cols[-1])))
assert ARM_COLUMNS["signal_plus_physio"] == B1_cols and ARM_COLUMNS["signal_plus_all"] == B1_cols + B2_cols
print("column order: physio == v0.2.1 B1_physio, all == v0.2.1 B3_physio_plus_acquisition: PASS")
print("age_imputed scaler (folds 1-8): mean={:.4f}, scale={:.4f}".format(
    float(age_scaler.mean_[0]), float(age_scaler.scale_[0])))


# =====================================================================================
# PART A - ENVIRONMENT REGRESSION CHECK (CPU)
# =====================================================================================
hdr("PART A - ENVIRONMENT REGRESSION CHECK (v0.3 LogisticRegression, CPU)")

import torch
VERSIONS = {"python": sys.version.split()[0], "numpy": np.__version__, "pandas": pd.__version__,
            "scipy": scipy.__version__, "sklearn": sklearn.__version__, "torch": torch.__version__,
            "torch_cuda": torch.version.cuda, "os": platform.platform()}
print("versions:", VERSIONS)

# v0.2.1 feature construction and LogisticRegression setup, verbatim, for B1 / B3 only
FEATURES_V021 = pd.concat([age_imputed.rename("age_imputed"), age_censored.rename("age_censored"),
                           sex.rename("sex"), device_dummies, site_dummies, nurse_dummies], axis=1)
y_train_full = lab.loc[mask_train]
regression_rows = []
for block_name, cols in [("B1_physio", B1_cols), ("B3_physio_plus_acquisition", B3_cols)]:
    Xtr_raw = FEATURES_V021.loc[mask_train, cols].to_numpy(dtype=float)
    Xev_raw = FEATURES_V021.loc[mask_eval, cols].to_numpy(dtype=float)
    block_cols = cols
    continuous_cols = [c for c in block_cols if c in ("age_imputed", "recording_year")]
    cont_idx = [block_cols.index(c) for c in continuous_cols]
    rest_idx = [i for i in range(len(block_cols)) if i not in cont_idx]
    scaler = StandardScaler().fit(Xtr_raw[:, cont_idx])
    Xtr_lr = np.empty_like(Xtr_raw)
    Xev_lr = np.empty_like(Xev_raw)
    Xtr_lr[:, cont_idx] = scaler.transform(Xtr_raw[:, cont_idx])
    Xev_lr[:, cont_idx] = scaler.transform(Xev_raw[:, cont_idx])
    if rest_idx:
        Xtr_lr[:, rest_idx] = Xtr_raw[:, rest_idx]
        Xev_lr[:, rest_idx] = Xev_raw[:, rest_idx]
    per_class = []
    for sc in SUPERCLASSES:
        model_lr = LogisticRegression(max_iter=2000, C=1.0, solver="lbfgs", random_state=SEED)
        model_lr.fit(Xtr_lr, y_train_full[sc].to_numpy())
        pos_col = list(model_lr.classes_).index(1)
        per_class.append(fast_auc(Y_EVAL[sc], model_lr.predict_proba(Xev_lr)[:, pos_col]))
    macro = float(np.nanmean(per_class))
    ref = V03_LR_MACRO_AUROC[block_name]
    ok = abs(macro - ref) <= ENV_REGRESSION_TOL
    regression_rows.append({"block": block_name, "learner": "LogisticRegression", "seed": SEED,
                            "macro_auroc_now": macro, "macro_auroc_v0.3": ref, "abs_diff": abs(macro - ref),
                            "tolerance": ENV_REGRESSION_TOL, "pass": ok, **VERSIONS})
    print("  {:28s} now={:.6f}  v0.3={:.6f}  |diff|={:.2e}  -> {}".format(
        block_name, macro, ref, abs(macro - ref), "PASS" if ok else "FAIL"))
env_reg_df = pd.DataFrame(regression_rows)
save_csv(env_reg_df, PREFIX + "env_regression.csv", index=False)
assert all(r["pass"] for r in regression_rows), \
    "ENVIRONMENT REGRESSION FAILED: v0.3 LogisticRegression results do not reproduce - aborting."
print("environment regression check: PASS")


# =====================================================================================
# GPU SETUP, SIGNAL CACHE (v0.4), LEAD NORMALIZATION
# =====================================================================================
hdr("GPU SETUP AND SIGNAL CACHE")

import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

if not torch.cuda.is_available():
    raise RuntimeError("torch.cuda.is_available() is False - no usable CUDA GPU in this kernel. Aborting.")
DEVICE = torch.device("cuda:0")


def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_all_seeds(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
GPU_NAME = torch.cuda.get_device_name(0)
print("gpu: {} | torch {} | CUDA {} | cuDNN {} | cudnn.deterministic=True, benchmark=False".format(
    GPU_NAME, torch.__version__, torch.version.cuda, torch.backends.cudnn.version()))

for path, expected in [(SIGNALS_NPY, EXPECTED_SIGNALS_NPY_SHA256), (SIGNALS_INDEX, EXPECTED_SIGNALS_INDEX_SHA256)]:
    assert os.path.isfile(path), "v0.4 cache file missing: " + path
    got = sha256_of(path)
    print("  {}  {}  -> {}".format(got, os.path.basename(path), "PASS" if got == expected else "FAIL"))
    assert got == expected, "v0.4 cache hash mismatch for " + path + " - aborting."
_old_idx = pd.read_csv(SIGNALS_INDEX, encoding="utf-8-sig")
assert list(_old_idx.columns) == list(current_index.columns) and _old_idx.astype(str).reset_index(drop=True).equals(
    current_index.astype(str).reset_index(drop=True)), "v0.4 sidecar index does not match folds 1-9 - aborting."
X = np.load(SIGNALS_NPY)
assert X.shape == (N_REC, N_LEADS, N_SAMPLES) and X.dtype == np.float32
print("signals loaded from the v0.4 cache: {} {} (folds 1-9 only)".format(X.shape, X.dtype))

# --- per-lead normalization parameters: folds 1-8 ONLY (two-pass, float64; verbatim v0.4) ---
CHUNK = 1000
_sum = np.zeros(N_LEADS, dtype=np.float64)
for s in range(0, len(train_rows), CHUNK):
    _sum += X[train_rows[s:s + CHUNK]].astype(np.float64).sum(axis=(0, 2))
_n_vals = len(train_rows) * N_SAMPLES
LEAD_MEAN = _sum / _n_vals
_sq = np.zeros(N_LEADS, dtype=np.float64)
for s in range(0, len(train_rows), CHUNK):
    xc = X[train_rows[s:s + CHUNK]].astype(np.float64) - LEAD_MEAN[None, :, None]
    _sq += (xc ** 2).sum(axis=(0, 2))
LEAD_STD = np.sqrt(_sq / _n_vals)
assert np.all(LEAD_STD > 0)
_v04_norm = pd.read_csv(V04_LEAD_NORM_CSV, encoding="utf-8-sig")
assert np.allclose(_v04_norm["mean"].to_numpy(), LEAD_MEAN, rtol=0, atol=1e-12) and \
    np.allclose(_v04_norm["std"].to_numpy(), LEAD_STD, rtol=0, atol=1e-12), \
    "lead normalization differs from v0.4_lead_norm_params.csv - aborting."
print("per-lead normalization (folds 1-8) recomputed and equal to v0.4_lead_norm_params.csv: PASS")
LEAD_MEAN_F32 = LEAD_MEAN.astype(np.float32)[:, None]
LEAD_STD_F32 = LEAD_STD.astype(np.float32)[:, None]


# =====================================================================================
# MODELS (verbatim from v0.4) + fusion check against dev_protocol.json
# =====================================================================================
# >>> MODELS
# SOURCE FOLLOWED: Strodthoff, Wagner, Schaeffter, Samek (2021) "Deep Learning for ECG
# Analysis: Benchmarks and Insights from PTB-XL", reference code repository
# helme/ecg_ptbxl_benchmarking: code/models/resnet1d.py (resnet1d_wang, BasicBlock1d,
# ResNet1d), code/models/inception1d.py (Inception1d, InceptionBlock1d, Shortcut1d),
# code/models/basic_conv1d.py (create_head1d, bn_drop_lin, AdaptiveConcatPool1d), and
# code/models/fastai_model.py (how the benchmark instantiates them). Re-implemented here
# from memory of that code in plain PyTorch (no fastai); the repository was NOT
# consulted at write time.
#
# DETAILS I AM UNSURE OF (should be checked against the repository):
#  1) resnet1d_wang kernel sizes: factory default kernel_size=[5,3], stem kernel 7,
#     stride_stem=1, no stem pooling, inplanes=128, layers [1,1,1], fix_feature_dim=True
#     (all stages 128 channels), stride 2 in stages 2 and 3. If fastai_model.py passes
#     the int kernel_size=5, BasicBlock1d expands it to [5, 5//2+1] = [5,3] -> same.
#  2) inception1d as used by the benchmark: kernel_size = 8*5 = 40 -> branch kernels
#     [39,19,9], depth 6, bottleneck 32, nb_filters 32 (output 4*32 = 128 channels),
#     residual shortcut every 3 blocks; all convs bias=False.
#  3) Head (create_head1d) with lin_ftrs_head=[128], ps_head=0.5, bn=True, relu,
#     concat pooling: AdaptiveConcatPool1d [max, avg] -> 2*nf -> BN, Dropout(0.25),
#     Linear(2*nf,128), ReLU -> BN, Dropout(0.5), Linear(128, 5). I believe the
#     benchmark default is lin_ftrs_head=[128]; with lin_ftrs_head=None the head would be
#     a single Linear(2*nf, 5) instead.
#  4) Order inside AdaptiveConcatPool1d is [max, avg] (fastai convention).
#  5) The reference aggregates sliding-window predictions with aggregate_fn="max" by
#     default in fastai_model.py (I am fairly but not fully sure); the project's dev
#     protocol uses the MEAN over windows (Part F). Flagged, not changed.
#  6) MaxPool1d(3, stride, padding=1) on the inception max-pool branch.
#
# METADATA FUSION (v0.4 PATCH FIX 1 -- sealed text is binding: "concatenated to the
# network's global-pooled embedding before the final linear classification head").
# The global-pooled embedding is the 2*nf = 256-dim AdaptiveConcatPool1d output; the
# "head" is the whole MLP head. The head's first block keeps the reference order
# (BN, Dropout(0.25), Linear, ReLU), split as: BN + Dropout applied to the embedding
# (`pool_norm`, identical across arms), then the RAW metadata (no BN, no dropout) is
# concatenated, then Linear(256 + meta_dim, 128) -> ReLU -> BN -> Dropout(0.5) ->
# Linear(128, 5). Backbone, pooling and pool_norm are identical across arms; the head's
# first Linear differs in input width by design.


def conv1d_same(in_planes, out_planes, kernel_size=3, stride=1):
    return nn.Conv1d(in_planes, out_planes, kernel_size=kernel_size, stride=stride,
                     padding=(kernel_size - 1) // 2, bias=False)


class BasicBlock1d(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, kernel_size=(3, 3), downsample=None):
        super().__init__()
        if isinstance(kernel_size, int):
            kernel_size = [kernel_size, kernel_size // 2 + 1]
        self.conv1 = conv1d_same(inplanes, planes, kernel_size=kernel_size[0], stride=stride)
        self.bn1 = nn.BatchNorm1d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv1d_same(planes, planes, kernel_size=kernel_size[1])
        self.bn2 = nn.BatchNorm1d(planes)
        self.downsample = downsample

    def forward(self, x):
        residual = x if self.downsample is None else self.downsample(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + residual)


def resnet1d_wang_backbone(input_channels=12, inplanes=128, kernel_size=(5, 3), kernel_size_stem=7,
                           stride_stem=1, layers=(1, 1, 1), stride=2):
    mods = [nn.Conv1d(input_channels, inplanes, kernel_size=kernel_size_stem, stride=stride_stem,
                      padding=(kernel_size_stem - 1) // 2, bias=False),
            nn.BatchNorm1d(inplanes), nn.ReLU(inplace=True)]          # no stem pooling
    cur = inplanes
    for i, n_blocks in enumerate(layers):
        st = 1 if i == 0 else stride
        planes = inplanes                                             # fix_feature_dim=True
        downsample = None
        if st != 1 or cur != planes:
            downsample = nn.Sequential(nn.Conv1d(cur, planes, kernel_size=1, stride=st, bias=False),
                                       nn.BatchNorm1d(planes))
        blocks = [BasicBlock1d(cur, planes, st, list(kernel_size), downsample)]
        cur = planes
        for _ in range(1, n_blocks):
            blocks.append(BasicBlock1d(cur, planes))
        mods.append(nn.Sequential(*blocks))
    return nn.Sequential(*mods), cur


class InceptionBlock1d(nn.Module):
    def __init__(self, ni, nb_filters, kss, stride=1, bottleneck_size=32):
        super().__init__()
        self.bottleneck = conv1d_same(ni, bottleneck_size, 1, stride) if bottleneck_size > 0 else nn.Identity()
        bi = bottleneck_size if bottleneck_size > 0 else ni
        self.convs = nn.ModuleList([conv1d_same(bi, nb_filters, ks) for ks in kss])
        self.conv_bottle = nn.Sequential(nn.MaxPool1d(3, stride, padding=1), conv1d_same(ni, nb_filters, 1))
        self.bn_relu = nn.Sequential(nn.BatchNorm1d((len(kss) + 1) * nb_filters), nn.ReLU())

    def forward(self, x):
        b = self.bottleneck(x)
        return self.bn_relu(torch.cat([c(b) for c in self.convs] + [self.conv_bottle(x)], dim=1))


class Shortcut1d(nn.Module):
    def __init__(self, ni, nf):
        super().__init__()
        self.act_fn = nn.ReLU(True)
        self.conv = conv1d_same(ni, nf, 1)
        self.bn = nn.BatchNorm1d(nf)

    def forward(self, inp, out):
        return self.act_fn(out + self.bn(self.conv(inp)))


class InceptionBackbone(nn.Module):
    def __init__(self, input_channels=12, kernel_size=40, depth=6, bottleneck_size=32, nb_filters=32,
                 use_residual=True):
        super().__init__()
        assert kernel_size >= 40 and depth % 3 == 0
        kss = [k - 1 if k % 2 == 0 else k for k in [kernel_size, kernel_size // 2, kernel_size // 4]]
        n_ks = len(kss) + 1
        self.depth, self.use_residual = depth, use_residual
        self.im = nn.ModuleList([InceptionBlock1d(input_channels if d == 0 else n_ks * nb_filters,
                                                  nb_filters, kss, bottleneck_size=bottleneck_size)
                                 for d in range(depth)])
        self.sk = nn.ModuleList([Shortcut1d(input_channels if d == 0 else n_ks * nb_filters, n_ks * nb_filters)
                                 for d in range(depth // 3)])
        self.out_channels = n_ks * nb_filters
        self.kss = kss

    def forward(self, x):
        input_res = x
        for d in range(self.depth):
            x = self.im[d](x)
            if self.use_residual and d % 3 == 2:
                x = self.sk[d // 3](input_res, x)
                input_res = x
        return x


class AdaptiveConcatPool1d(nn.Module):
    def __init__(self):
        super().__init__()
        self.mp = nn.AdaptiveMaxPool1d(1)
        self.ap = nn.AdaptiveAvgPool1d(1)

    def forward(self, x):
        return torch.cat([self.mp(x), self.ap(x)], dim=1).flatten(1)


class ECGNet(nn.Module):
    """Backbone + reference head. Metadata (meta_dim >= 0) is concatenated, raw, to the
    global-pooled embedding (after its BN + Dropout) and enters the head's first Linear.
    backbone / pool / pool_norm are built in the same order with the same shapes for
    every arm, so with the same seed they are initialised identically (checked below)."""

    def __init__(self, arch, meta_dim, n_classes=5, input_channels=12, lin_ftrs=128, ps_head=0.5):
        super().__init__()
        if arch == "resnet1d_wang":
            self.backbone, nf = resnet1d_wang_backbone(input_channels=input_channels)
        elif arch == "inception1d":
            self.backbone = InceptionBackbone(input_channels=input_channels, kernel_size=40)
            nf = self.backbone.out_channels
        else:
            raise ValueError(arch)
        self.pool = AdaptiveConcatPool1d()
        # embedding pre-processing: identical across arms
        self.pool_norm = nn.Sequential(nn.BatchNorm1d(2 * nf), nn.Dropout(ps_head / 2))
        self.embedding_dim = 2 * nf
        self.meta_dim = meta_dim
        # head: receives [embedding (+ raw metadata)]
        self.hidden = nn.Sequential(nn.Linear(2 * nf + meta_dim, lin_ftrs), nn.ReLU(inplace=True),
                                    nn.BatchNorm1d(lin_ftrs), nn.Dropout(ps_head))
        self.final = nn.Linear(lin_ftrs, n_classes)

    def forward(self, x, meta=None):
        z = self.pool_norm(self.pool(self.backbone(x)))
        if self.meta_dim > 0:
            z = torch.cat([z, meta], dim=1)   # raw metadata: no BN, no dropout
        return self.final(self.hidden(z))
# <<< MODELS


METADATA_FUSION_DESCRIPTION = (
    "location: the metadata vector is concatenated to the network's global-pooled embedding (the "
    "2*nf = 256-dim AdaptiveConcatPool1d [max, avg] output, after that embedding's BatchNorm1d + "
    "Dropout(0.25)) and the concatenation is the input of the head's first Linear "
    "(256 + meta_dim -> 128), followed by ReLU, BatchNorm1d(128), Dropout(0.5), Linear(128 -> 5); "
    "metadata enters RAW (age_imputed standardized with training-fold statistics, all other columns "
    "0/1): no BatchNorm and no dropout is applied to the metadata; identical across arms: backbone, "
    "global pooling and the embedding's BatchNorm + Dropout (same seed -> identical initial weights, "
    "asserted in v0.4); differs across arms by design: only the input width of the head's first "
    "Linear (256 / 259 / 338 for signal_only / signal_plus_physio / signal_plus_all)")
assert METADATA_FUSION_DESCRIPTION == DEV["metadata_fusion"], \
    "ECGNet fusion description differs from the one locked in dev_protocol.json - aborting."
print("ECGNet metadata fusion == dev_protocol.json 'metadata_fusion': PASS")


class RandomCropDataset(Dataset):
    """Training records -> random 250-sample crop, per-lead normalized with folds 1-8 stats."""

    def __init__(self, X, rows, meta, y, crop, seed):
        self.X, self.rows, self.meta, self.y, self.crop = X, rows, meta, y, crop
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        s = int(self.rng.integers(0, N_SAMPLES - self.crop + 1))
        x = (self.X[r, :, s:s + self.crop] - LEAD_MEAN_F32) / LEAD_STD_F32
        return (torch.from_numpy(np.ascontiguousarray(x, dtype=np.float32)),
                torch.from_numpy(self.meta[r]), torch.from_numpy(self.y[r]))


# =====================================================================================
# TRAINING / EVALUATION DEFINITION  ->  v0.5_training_details.json (write-once)
# =====================================================================================
hdr("TRAINING / EVALUATION DEFINITION")

ADAMW_BETAS = (0.9, 0.99)
ADAMW_EPS = 1e-8
ONECYCLE = {"pct_start": 0.3, "div_factor": 25.0, "final_div_factor": 1e4, "anneal_strategy": "cos",
            "cycle_momentum": True, "base_momentum": 0.85, "max_momentum": 0.95, "three_phase": False}

TRAINING_DETAILS = {
    "optimizer": {"name": "torch.optim.AdamW", "betas": list(ADAMW_BETAS), "eps": ADAMW_EPS,
                  "weight_decay": WEIGHT_DECAY, "weight_decay_source": "dev_protocol.json",
                  "applied_to": "all parameters (incl. BatchNorm and biases)"},
    "lr_scheduler": {"name": "torch.optim.lr_scheduler.OneCycleLR", "max_lr": "the run's lr",
                     "total_steps": "epochs * len(train_loader)", "stepped": "every batch (after optimizer.step())",
                     **ONECYCLE,
                     "momentum_note": ("cycle_momentum/base_momentum/max_momentum are torch defaults: AdamW "
                                       "beta1 is cycled 0.95 -> 0.85 -> 0.95 in the opposite phase of the lr")},
    "epochs": EPOCHS, "epochs_source": "dev_protocol.json",
    "batch_size": BATCH_SIZE, "batch_size_source": "dev_protocol.json",
    "train_data": ("folds 1-8; RandomCropDataset (v0.4): random {}-sample crop per record per epoch, start "
                   "uniform over [0, {}], per-lead normalized with folds 1-8 statistics".format(CROP, N_SAMPLES - CROP)),
    "data_order": ("DataLoader(shuffle=True, generator=torch.Generator().manual_seed(run_seed), num_workers=0, "
                   "pin_memory=True, drop_last=False); RandomCropDataset rng = np.random.default_rng(run_seed)"),
    "seeding": ("set_all_seeds(run_seed) (python random, numpy, torch, torch.cuda) immediately before model "
                "construction; cudnn.deterministic=True, cudnn.benchmark=False"),
    "training_loss": "BCEWithLogitsLoss (mean over batch and the 5 classes)",
    "training_loss_logging": ("per-epoch MEAN TRAINING loss = sample-weighted mean of batch losses over the "
                              "epoch (training data only), saved per run"),
    "validation": ("NO validation metric is computed at any epoch other than the end of epoch {}; no "
                   "checkpoint selection; the model after the last optimizer step is evaluated".format(EPOCHS)),
    "evaluation": ("model.eval(); torch.no_grad(); {} windows (starts {}) of {} samples per fold-9 record, "
                   "per-lead normalized with folds 1-8 statistics; per-window sigmoid PROBABILITIES averaged "
                   "per record (mean of probabilities, not of logits); metadata identical for all windows of a "
                   "record; inference batch size {} (no effect on results)".format(
                       len(WINDOW_STARTS), WINDOW_STARTS, CROP, EVAL_BATCH_SIZE)),
    "metric": ("macro-AUROC = mean over NORM, MI, STTC, CD, HYP of fast_auc (v0.2.1 function); verified once "
               "against sklearn.metrics.roc_auc_score (abs tol {})".format(SKLEARN_CHECK_TOL)),
    "lr_tie_break": ("highest fold-9 macro-AUROC; if two lrs are within {} of each other, choose the SMALLER lr "
                     "(implemented as: among lrs with macro-AUROC >= best - {}, take the smallest)".format(
                         LR_TIE_TOL, LR_TIE_TOL)),
    "per_run_outputs": ("v0.5_{arch}_{arm}_seed{seed}_lr{lr}_fold9_probs.csv (ecg_id, patient_id, 5 "
                        "probabilities), _train_loss.csv (per-epoch mean training loss), _state_dict.pt "
                        "(final model); all hashed in the manifest"),
    "determinism_note": ("torch.use_deterministic_algorithms is NOT enabled; some CUDA kernels may not be "
                         "bitwise reproducible across re-runs"),
}

if TRAINING_DETAILS_EXISTS:
    assert plain(TRAINING_DETAILS) == TRAINING_DETAILS_ON_DISK["details"], \
        "v0.5_training_details.json on disk differs from this code's training definition - aborting."
    assert TRAINING_DETAILS_ON_DISK["parent_seal_self_sha256"] == EXPECTED_SEAL_SHA256
    assert TRAINING_DETAILS_ON_DISK["parent_dev_protocol_self_sha256"] == EXPECTED_DEV_PROTOCOL_SHA256
    print("v0.5_training_details.json already exists and matches this code exactly (self_sha256 {}).".format(
        TRAINING_DETAILS_SHA))
else:
    TRAINING_DETAILS_SHA = write_once_json(TRAINING_DETAILS_PATH, {
        "document": "v0.5 training / evaluation implementation details (not specified in dev_protocol.json)",
        "timestamp": datetime.now().astimezone().isoformat(),
        "code_version": CODE_VERSION,
        "parent_seal_self_sha256": EXPECTED_SEAL_SHA256,
        "parent_dev_protocol_self_sha256": EXPECTED_DEV_PROTOCOL_SHA256,
        "written_before_first_training_run": True,
        "details": TRAINING_DETAILS,
        "self_sha256_procedure": SELF_SHA256_PROCEDURE,
        "write_once": True,
    })
    print("[saved] {} (self_sha256 {}, verified) -- written BEFORE the first training run".format(
        TRAINING_DETAILS_PATH, TRAINING_DETAILS_SHA))
OUTPUT_FILES.append(TRAINING_DETAILS_PATH)


def run_name(arch, arm, seed, lr):
    return "{}{}_{}_seed{}_lr{:g}".format(PREFIX, arch, arm, seed, lr)


def run_paths(arch, arm, seed, lr):
    base = os.path.join(OUT_DIR, run_name(arch, arm, seed, lr))
    return {"probs": base + "_fold9_probs.csv", "loss": base + "_train_loss.csv", "state": base + "_state_dict.pt"}


_SKLEARN_VERIFIED = {"done": False}


def macro_auroc_from_probs(probs):
    """probs: (n_fold9, 5) in eval_rows order. Returns (macro, per-class dict)."""
    per = {sc: fast_auc(Y_EVAL[sc], probs[:, k]) for k, sc in enumerate(SUPERCLASSES)}
    if not _SKLEARN_VERIFIED["done"]:
        for k, sc in enumerate(SUPERCLASSES):
            sk = float(roc_auc_score(Y_EVAL[sc], probs[:, k]))
            assert abs(sk - per[sc]) <= SKLEARN_CHECK_TOL, "fast_auc != sklearn roc_auc_score for " + sc
        _SKLEARN_VERIFIED["done"] = True
        print("    fast_auc verified against sklearn.metrics.roc_auc_score (abs tol {}): PASS".format(SKLEARN_CHECK_TOL))
    return float(np.mean([per[sc] for sc in SUPERCLASSES])), per


@torch.no_grad()
def predict_fold9(model, arm):
    """Mean over the 7 windows of per-window sigmoid probabilities, per fold-9 record."""
    model.eval()
    probs = np.zeros((len(eval_rows), len(SUPERCLASSES)), dtype=np.float64)
    meta_all = META[arm]
    for st in WINDOW_STARTS:
        for b in range(0, len(eval_rows), EVAL_BATCH_SIZE):
            rows = eval_rows[b:b + EVAL_BATCH_SIZE]
            xw = (X[rows, :, st:st + CROP] - LEAD_MEAN_F32) / LEAD_STD_F32
            xb = torch.from_numpy(np.ascontiguousarray(xw, dtype=np.float32)).to(DEVICE)
            mb = torch.from_numpy(meta_all[rows]).to(DEVICE)
            probs[b:b + len(rows)] += torch.sigmoid(model(xb, mb)).double().cpu().numpy()
    return probs / len(WINDOW_STARTS)


def save_run_outputs(paths, probs, loss_rows, model):
    pdf = pd.DataFrame({"ecg_id": current_index["ecg_id"].to_numpy()[eval_rows],
                        "patient_id": current_index["patient_id"].to_numpy()[eval_rows]})
    for k, sc in enumerate(SUPERCLASSES):
        pdf["prob_" + sc] = probs[:, k]
    pdf.to_csv(paths["probs"], index=False, encoding="utf-8-sig")
    pd.DataFrame(loss_rows).to_csv(paths["loss"], index=False, encoding="utf-8-sig")
    torch.save(model.state_dict(), paths["state"])
    for p in (paths["probs"], paths["loss"], paths["state"]):
        OUTPUT_FILES.append(p)


def load_saved_probs(paths):
    pdf = pd.read_csv(paths["probs"], encoding="utf-8-sig")
    assert (pdf["ecg_id"].to_numpy() == current_index["ecg_id"].to_numpy()[eval_rows]).all()
    return pdf[["prob_" + sc for sc in SUPERCLASSES]].to_numpy(dtype=np.float64)


def train_run(arch, arm, seed, lr):
    """Train 50 epochs on folds 1-8, evaluate ONCE on fold 9 at the end, save run files."""
    assert arm != FORBIDDEN_ARM, "signal_plus_all must not be trained in v0.5 - aborting."
    assert TRAINING_DETAILS_SHA is not None and os.path.isfile(TRAINING_DETAILS_PATH), \
        "training details must be written before any training run"
    bs = BATCH_SIZE[arch]
    assert len(train_rows) % bs != 1, "last training batch would have size 1 (BatchNorm) - aborting."
    set_all_seeds(seed)
    model = ECGNet(arch, META[arm].shape[1]).to(DEVICE)
    ds = RandomCropDataset(X, train_rows, META[arm], Y, CROP, seed)
    loader = DataLoader(ds, batch_size=bs, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True,
                        drop_last=False, generator=torch.Generator().manual_seed(seed))
    opt = torch.optim.AdamW(model.parameters(), lr=lr, betas=ADAMW_BETAS, eps=ADAMW_EPS, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=EPOCHS * len(loader), **ONECYCLE)
    crit = nn.BCEWithLogitsLoss()
    loss_rows = []
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    for epoch in range(1, EPOCHS + 1):
        model.train()
        loss_sum = torch.zeros((), device=DEVICE, dtype=torch.float64)
        n_seen = 0
        for xb, mb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            mb = mb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            loss = crit(model(xb, mb), yb)
            loss.backward()
            opt.step()
            sched.step()
            loss_sum += loss.detach().double() * xb.shape[0]
            n_seen += xb.shape[0]
        mean_loss = float(loss_sum.item()) / n_seen
        assert math.isfinite(mean_loss), "{}: non-finite training loss at epoch {} - aborting.".format(
            run_name(arch, arm, seed, lr), epoch)
        loss_rows.append({"epoch": epoch, "mean_train_loss": mean_loss, "lr_end_of_epoch": sched.get_last_lr()[0]})
        if epoch % 10 == 0 or epoch == 1:
            print("    epoch {:2d}/{} | mean training loss {:.4f} | {:.0f}s".format(
                epoch, EPOCHS, mean_loss, time.time() - t0))
    torch.cuda.synchronize()
    wall = time.time() - t0
    peak_mb = torch.cuda.max_memory_allocated() / 1024 ** 2
    probs = predict_fold9(model, arm)           # the ONE fold-9 evaluation, end of epoch 50
    macro, per = macro_auroc_from_probs(probs)
    save_run_outputs(run_paths(arch, arm, seed, lr), probs, loss_rows, model)
    del model, opt, sched, loader
    torch.cuda.empty_cache()
    return {"arch": arch, "arm": arm, "seed": seed, "lr": lr, "fold9_macro_auroc": macro,
            **{"auroc_" + sc: per[sc] for sc in SUPERCLASSES},
            "final_epoch_train_loss": loss_rows[-1]["mean_train_loss"],
            "wall_time_s": round(wall, 1), "peak_vram_mb": round(peak_mb, 1)}


# =====================================================================================
# PART B - LR SELECTION (signal_plus_physio, seed 42)
# =====================================================================================
hdr("PART B - LR SELECTION (signal_plus_physio, seed {})".format(SEED))
SEL_ARM = "signal_plus_physio"
TABLE_CSV = os.path.join(OUT_DIR, PREFIX + "lr_selection_table.csv")

if LR_SELECTION_EXISTS:
    lr_table = pd.DataFrame(LR_SEL["table"])
    print("Part B SKIPPED: lr_selection.json exists (self_sha256 {}). Table from the file:".format(LR_SELECTION_SHA))
    print(lr_table.round(4).to_string(index=False))
    print("selected lr:", SELECTED_LR)
    for arch in ARCHS:
        for lr in LR_GRID:
            for p in run_paths(arch, SEL_ARM, SEED, lr).values():
                if os.path.isfile(p):
                    OUTPUT_FILES.append(p)
    if os.path.isfile(TABLE_CSV):
        OUTPUT_FILES.append(TABLE_CSV)
else:
    table_rows = []
    for arch in ARCHS:
        for lr in LR_GRID:
            print("  training {} | {} | seed {} | lr {:g} | {} epochs | batch {}".format(
                arch, SEL_ARM, SEED, lr, EPOCHS, BATCH_SIZE[arch]))
            row = train_run(arch, SEL_ARM, SEED, lr)
            table_rows.append(row)
            print("  -> fold-9 macro-AUROC {:.4f} | {:.0f}s | peak VRAM {:.0f} MB".format(
                row["fold9_macro_auroc"], row["wall_time_s"], row["peak_vram_mb"]))
    lr_table = pd.DataFrame(table_rows)
    print("\n--- lr selection table (fold-9, end of epoch {}) ---".format(EPOCHS))
    print(lr_table.drop(columns=["arm", "seed"]).round(4).to_string(index=False))

    SELECTED_LR, selection_detail = {}, {}
    for arch in ARCHS:
        sub = lr_table[lr_table["arch"] == arch]
        best = float(sub["fold9_macro_auroc"].max())
        candidates = sub[sub["fold9_macro_auroc"] >= best - LR_TIE_TOL]
        chosen = float(candidates["lr"].min())
        SELECTED_LR[arch] = chosen
        selection_detail[arch] = {"best_macro_auroc": best, "lrs_within_tie_tolerance": sorted(candidates["lr"].tolist()),
                                  "selected_lr": chosen}
        print("  [{}] best macro-AUROC {:.4f}; lrs within {}: {} -> selected lr {:g}".format(
            arch, best, LR_TIE_TOL, sorted(candidates["lr"].tolist()), chosen))
    lr_table.to_csv(TABLE_CSV, index=False, encoding="utf-8-sig")
    OUTPUT_FILES.append(TABLE_CSV)
    print("[saved] " + TABLE_CSV)

    LR_SELECTION_SHA = write_once_json(LR_SELECTION_PATH, {
        "document": "learning-rate selection for H3 development (write-once)",
        "timestamp": datetime.now().astimezone().isoformat(),
        "code_version": CODE_VERSION,
        "parent_hashes": {"pre_registration": EXPECTED_SEAL_SHA256,
                          "dev_protocol": EXPECTED_DEV_PROTOCOL_SHA256,
                          "training_details": TRAINING_DETAILS_SHA},
        "arm": SEL_ARM, "seed": SEED, "epochs": EPOCHS, "lr_grid": LR_GRID,
        "table": lr_table.to_dict(orient="records"),
        "selected_lr": SELECTED_LR,
        "selection_detail": selection_detail,
        "selection_rule_from_dev_protocol": SELECTION_RULE_TEXT,
        "tie_break_from_training_details": TRAINING_DETAILS["lr_tie_break"],
        "self_sha256_procedure": SELF_SHA256_PROCEDURE,
        "write_once": True,
    })
    print("[saved] {} (self_sha256 {}, verified) -- Part C may start now.".format(LR_SELECTION_PATH, LR_SELECTION_SHA))
OUTPUT_FILES.append(LR_SELECTION_PATH)
assert os.path.isfile(LR_SELECTION_PATH) and SELECTED_LR is not None


# =====================================================================================
# PART C - PIPELINE SANITY GATE (signal_only, resnet1d_wang, seed 42, selected lr)
# =====================================================================================
hdr("PART C - PIPELINE SANITY GATE")
GATE_ARCH, GATE_ARM = "resnet1d_wang", "signal_only"
gate_lr = SELECTED_LR[GATE_ARCH]
gate_paths = run_paths(GATE_ARCH, GATE_ARM, SEED, gate_lr)
if all(os.path.isfile(p) for p in gate_paths.values()):
    print("Part C run files already exist -> gate re-evaluated from the SAVED fold-9 probabilities; "
          "signal_only is NOT retrained.")
    gate_macro, gate_per = macro_auroc_from_probs(load_saved_probs(gate_paths))
    OUTPUT_FILES.extend(gate_paths.values())
    gate_row = {"arch": GATE_ARCH, "arm": GATE_ARM, "seed": SEED, "lr": gate_lr, "fold9_macro_auroc": gate_macro,
                **{"auroc_" + sc: gate_per[sc] for sc in SUPERCLASSES}, "reused_saved_probabilities": True}
else:
    print("  training {} | {} | seed {} | selected lr {:g} | {} epochs".format(GATE_ARCH, GATE_ARM, SEED, gate_lr, EPOCHS))
    gate_row = train_run(GATE_ARCH, GATE_ARM, SEED, gate_lr)
    gate_row["reused_saved_probabilities"] = False
    gate_macro = gate_row["fold9_macro_auroc"]
GATE_PASS = gate_macro >= SANITY_GATE_THRESHOLD
print("\n  signal_only resnet1d_wang fold-9 macro-AUROC = {:.4f} | per class: {}".format(
    gate_macro, {sc: round(gate_row["auroc_" + sc], 4) for sc in SUPERCLASSES}))
print("SANITY GATE: {}".format("PASS" if GATE_PASS else "FAIL"))

_phys = lr_table[(lr_table["arch"] == GATE_ARCH) & (lr_table["lr"] == gate_lr)]
assert len(_phys) == 1
EXPLORATORY = {"label": "EXPLORATORY -- not part of H3", "arch": GATE_ARCH, "seed": SEED, "lr": gate_lr,
               "signal_only_macro_auroc": gate_macro,
               "signal_plus_physio_macro_auroc": float(_phys["fold9_macro_auroc"].iloc[0])}
print("EXPLORATORY -- not part of H3: resnet1d_wang, seed {}, lr {:g}: signal_only {:.4f} | "
      "signal_plus_physio {:.4f}".format(SEED, gate_lr, EXPLORATORY["signal_only_macro_auroc"],
                                         EXPLORATORY["signal_plus_physio_macro_auroc"]))


# =====================================================================================
# MANIFEST
# =====================================================================================
hdr("MANIFEST")
seen, outputs = set(), []
for p in OUTPUT_FILES:
    if p in seen:
        continue
    seen.add(p)
    outputs.append({"file": p, "bytes": os.path.getsize(p), "sha256": sha256_of(p)})
manifest = {
    "code_version": CODE_VERSION,
    "run_started": RUN_STARTED,
    "run_finished": datetime.now().astimezone().isoformat(),
    "versions": VERSIONS, "gpu": GPU_NAME,
    "input_files_sha256": INPUT_HASHES,
    "parent_hashes": {"pre_registration": EXPECTED_SEAL_SHA256, "dev_protocol": EXPECTED_DEV_PROTOCOL_SHA256,
                      "training_details": TRAINING_DETAILS_SHA, "lr_selection": LR_SELECTION_SHA,
                      "v0.4_signals_npy": EXPECTED_SIGNALS_NPY_SHA256, "v0.4_signals_index": EXPECTED_SIGNALS_INDEX_SHA256},
    "recipe_from_dev_protocol": {"epochs": EPOCHS, "lr_grid": LR_GRID, "weight_decay": WEIGHT_DECAY,
                                 "batch_size": BATCH_SIZE, "crop": CROP, "eval_stride": EVAL_STRIDE,
                                 "window_starts": WINDOW_STARTS, "aggregation": AGGREGATION},
    "env_regression": regression_rows,
    "part_b_skipped": LR_SELECTION_EXISTS,
    "lr_selection_table": plain(lr_table.to_dict(orient="records")),
    "selected_lr": SELECTED_LR,
    "sanity_gate": {"threshold": SANITY_GATE_THRESHOLD, "result": plain(gate_row), "pass": bool(GATE_PASS)},
    "exploratory": EXPLORATORY,
    "signal_plus_all_trained": False,
    "fold10_opened": False,
    "warnings": WARNINGS,
    "outputs": outputs,
}
MANIFEST_PATH = os.path.join(OUT_DIR, PREFIX + "manifest.json")
with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2, default=_json_default)
for o in outputs:
    print("  {}  {}".format(o["sha256"], o["file"]))
print("[saved] " + MANIFEST_PATH)
print("manifest sha256:", sha256_of(MANIFEST_PATH))


# =====================================================================================
# FINAL SUMMARY
# =====================================================================================
hdr("SUMMARY - " + CODE_VERSION)
print("environment regression     : PASS (scipy {}, sklearn {}, torch {})".format(
    VERSIONS["scipy"], VERSIONS["sklearn"], VERSIONS["torch"]))
print("training details           :", TRAINING_DETAILS_SHA)
print("lr_selection               : {} | selected lr {}".format(LR_SELECTION_SHA, SELECTED_LR))
print("sanity gate                : {:.4f} -> {}".format(gate_macro, "PASS" if GATE_PASS else "FAIL"))
print("\nwarnings ({}):".format(len(WARNINGS)))
for w in WARNINGS:
    print("  - " + w)
if not GATE_PASS:
    print("\nSANITY GATE: FAIL -- stopping here (everything above is saved). Do not try other settings; "
          "report and debug.")
print("\nsignal_plus_all was not trained in v0.5. Fold 10 was not opened.")
if not GATE_PASS:
    raise SystemExit


STEP 0 - INTEGRITY
pre_registration.json : self_sha256 5183f54569f3358a5339b16a2c3c3be49f5c7798eabe6b0bb25946119c2ba469 -> PASS
dev_protocol.json     : self_sha256 c967ac38c8cc9382bf41fdd3785c8bbd730db196e349846b8052d856bbac5fc6 -> PASS
recipe from dev_protocol.json: epochs=50 | lr grid=[0.001, 0.003, 0.01] | weight_decay=0.01 | batch sizes={'resnet1d_wang': 128, 'inception1d': 128} | crop=250 | eval windows 250 stride 125 starts [0, 125, 250, 375, 500, 625, 750] | aggregation=mean -> all equal the constants: PASS
input CSV hashes      : equal to v0.4 -> PASS
lr_selection.json not present -> Part B will run and write it once.

DATA, LABELS, METADATA
records: folds 1-8 = 17418 | fold 9 = 2183 | fold-10 records in source CSV, never opened: 2198
fold-9 positives: {'NORM': 955, 'MI': 540, 'STTC': 528, 'CD': 495, 'HYP': 268} -> PASS
age: sentinel 300 affects 259 records; flagged via age_censored, imputed with 90.
  signal_only          metadata dim =   0
  signal_plus_physio   metadata dim